# Step 3 — Model Exploration

We test the recommender interactively here before the logic moves into `src/recommender.py`.

**How it works:**
1. Look up the query game(s) in the feature matrix
2. Average their vectors (supports multi-game queries)
3. Compute cosine similarity against all other games
4. Return the top N

In [10]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

os.chdir(r"c:\Users\diogo\steam-recommender")

with open("data/features.pkl", "rb") as f:
    data = pickle.load(f)

games = data["games"]
matrix = data["matrix"]
vocab = data["vocab"]

name_to_idx = {name.lower(): i for i, name in enumerate(games["name"])}

print(f"Loaded {len(games)} games, {matrix.shape[1]} tag features")

Loaded 4995 games, 380 tag features


## Core recommender function

In [11]:
def find_game_idx(query: str) -> int | None:
    """Find game index by exact or partial name match."""
    q = query.lower().strip()
    if q in name_to_idx:
        return name_to_idx[q]
    # partial match — return first hit
    for name, idx in name_to_idx.items():
        if q in name:
            return idx
    return None


def recommend(query_games: str | list[str], n: int = 10) -> pd.DataFrame:
    """
    query_games: one game name or a list of game names
    Returns a DataFrame of the top N recommended games.
    """
    if isinstance(query_games, str):
        query_games = [query_games]

    query_indices = []
    for name in query_games:
        idx = find_game_idx(name)
        if idx is None:
            print(f"Warning: '{name}' not found — skipping")
        else:
            matched = games.loc[idx, 'name']
            if matched.lower() != name.lower():
                print(f"Matched '{name}' → '{matched}'")
            query_indices.append(idx)

    if not query_indices:
        return pd.DataFrame()

    # Average the query vectors
    query_vec = np.asarray(matrix[query_indices].mean(axis=0))
    scores = cosine_similarity(query_vec, matrix)[0]

    # Exclude the query games themselves
    scores[query_indices] = -1

    top_idx = np.argsort(scores)[::-1][:n]

    results = games.loc[top_idx, ["appid", "name", "developer", "top_tags"]].copy()
    results["similarity"] = scores[top_idx].round(4)
    return results.reset_index(drop=True)

## Try it out — single game query

In [12]:
recommend("Counter-Strike 2")

""


In [13]:
recommend("The Witcher 3: Wild Hunt")

,appid,name,developer,top_tags,similarity
0,20900,The Witcher: Enhanced Edition Director's Cut,CD PROJEKT RED,"RPG,Fantasy,Story Rich,Mature,Singleplayer",0.8350
1,20920,The Witcher 2: Assassins of Kings Enhanced Edi...,CD PROJEKT RED,"RPG,Fantasy,Mature,Story Rich,Choices Matter",0.8271
2,208730,Game of Thrones,Cyanide Studios,"RPG,Fantasy,Action,Story Rich,Third Person",0.7105
3,204030,Fable - The Lost Chapters,Lionhead Studios,"RPG,Adventure,Action,Fantasy,Third Person",0.6871
4,1930,Two Worlds Epic Edition,Reality Pump Studios,"RPG,Open World,Fantasy,Singleplayer,Third Person",0.6481
5,22330,The Elder Scrolls IV: Oblivion Game of the Yea...,Bethesda Game Studios,"RPG,Open World,Fantasy,Singleplayer,Moddable",0.6477
6,900883,The Elder Scrolls IV: Oblivion Game of the Yea...,Bethesda Game Studios®,"RPG,Open World,Fantasy,Singleplayer,Moddable",0.6477
7,39500,Gothic 3,Piranha Bytes,"RPG,Open World,Fantasy,Action,Singleplayer",0.6371
8,253980,Enclave,Starbreeze,"RPG,Action,Fantasy,Third Person,Hack and Slash",0.6293
9,606880,GreedFall,Spiders,"RPG,Open World,Character Customization,Singlep...",0.6285


In [14]:
recommend("Stardew Valley")

,appid,name,developer,top_tags,similarity
0,1158160,Coral Island,Stairway Games,"Farming Sim,Life Sim,Relaxing,Dating Sim,Simul...",0.7992
1,894940,Littlewood,Sean Young,"Pixel Graphics,Life Sim,City Builder,Simulatio...",0.7809
2,1432860,Sun Haven,Pixel Sprout Studios,"Farming Sim,Pixel Graphics,Multiplayer,Life Si...",0.7027
3,2142790,Fields of Mistria,NPC Studio,"Early Access,Farming Sim,Dating Sim,RPG,Life Sim",0.6825
4,666140,My Time at Portia,Pathea Games,"Life Sim,RPG,Open World,Farming Sim,Crafting",0.6815
5,1062520,Dinkum,James Bendon,"Open World Survival Craft,Agriculture,Farming ...",0.6792
6,1868140,DAVE THE DIVER,MINTROCKET,"Pixel Graphics,Casual,Management,Adventure,Rel...",0.6478
7,1658040,I Am Future: Cozy Apocalypse Survival,Mandragora,"Casual,Singleplayer,Farming Sim,RPG,Agriculture",0.6358
8,751780,Forager,HopFrog,"Open World Survival Craft,Survival,Pixel Graph...",0.6099
9,405710,Staxel,Plukit,"Farming Sim,Character Customization,Cute,Sandb...",0.6079


## Multi-game query

Pass a list of games you like — the recommender averages their feature vectors and finds what sits in between.

In [15]:
recommend(["Counter-Strike 2", "PUBG: BATTLEGROUNDS", "Left 4 Dead 2"], n=10)

,appid,name,developer,top_tags,similarity
0,500,Left 4 Dead,Valve,"Zombies,Co-op,Multiplayer,FPS,Action",0.7940
1,493520,GTFO,10 Chambers,"Online Co-Op,Horror,Co-op,FPS,Difficult",0.6921
2,15120,Tom Clancy's Rainbow Six Vegas 2,Ubisoft Montreal,"Tactical,Action,FPS,Co-op,Singleplayer",0.6719
3,232090,Killing Floor 2,Tripwire Interactive,"Zombies,Multiplayer,Online Co-Op,Gore,FPS",0.6605
4,813820,Realm Royale,Heroic Leap Games,"Battle Royale,Free to Play,Multiplayer,Shooter...",0.6509
5,518150,Intruder,Superboss Games,"Multiplayer,Stealth,Tactical,First-Person,Team...",0.6273
6,273350,Evolve Stage 2,Turtle Rock Studios,"Action,Multiplayer,Co-op,FPS,Online Co-Op",0.6221
7,460930,Tom Clancy's Ghost Recon Wildlands,"Ubisoft Paris, Ubisoft Annecy, Ubisoft Buchare...","Open World,Shooter,Co-op,Stealth,Multiplayer",0.6186
8,433850,Z1 Battle Royale,Daybreak Game Company,"Survival,Massively Multiplayer,Multiplayer,Ope...",0.6143
9,222880,Insurgency,New World Interactive,"FPS,Realistic,Tactical,Multiplayer,Shooter",0.6134


## Search for a game name

Useful when you're not sure of the exact title.

In [17]:
def search(query: str, n: int = 10) -> pd.DataFrame:
    q = query.lower()
    mask = games["name"].str.lower().str.contains(q, regex=False)
    return games[mask][["appid", "name", "developer", "top_tags"]].head(n)

search("counter")

,appid,name,developer,top_tags
0,730,Counter-Strike: Global Offensive,Valve,"FPS,Shooter,Multiplayer,Competitive,Action"
63,10,Counter-Strike,Valve,"Action,FPS,Multiplayer,Shooter,Classic"
64,240,Counter-Strike: Source,Valve,"Shooter,FPS,Action,Multiplayer,Team-Based"
74,100,Counter-Strike: Condition Zero,Valve,"Action,FPS,Shooter,Multiplayer,First-Person"
118,80,Counter-Strike: Condition Zero,Valve,"Action,FPS,Shooter,Multiplayer,First-Person"
195,273110,Counter-Strike Nexon: Studio,"Valve, NEXON","Free to Play,FPS,Zombies,Sandbox,PvE"
893,41000,Serious Sam HD: The First Encounter,Croteam,"FPS,Action,Shooter,Co-op,Gore"
3057,41050,Serious Sam Classic: The First Encounter,Croteam,"Action,FPS,Classic,Shooter,Gore"
3172,385270,Jet Racing Extreme: The First Encounter,Real Dynamics,"Racing,Action,Indie,Simulation,Sports"
3250,201480,Serious Sam: The Random Encounter,"Vlambeer, Croteam","Action,Indie,RPG,Turn-Based,2D"


---
## Summary

The recommender works. The logic in this notebook lives in `src/recommender.py`.

**Next**: Run `uvicorn src.api:app --reload` to start the REST API.